In [1]:
import cv2
from yolo_prediction import YOLO_Pred

In [2]:
yolo=YOLO_Pred('./Model/weights/best.onnx','data.yaml')

In [8]:
img =cv2.imread('./sample.jpg')
cv2.imshow('img',img)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [10]:
#prediction
img_pred=yolo.prediction(img)

In [11]:

cv2.imshow('prediction image',img_pred)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [1]:
import cv2
import numpy as np
import yaml
from yaml.loader import SafeLoader

class YOLO_Pred():
    def __init__(self, onnx_model, data_yaml):
        # Load class names and number of classes from the YAML file
        with open(data_yaml, mode='r') as f:
            data_yaml = yaml.load(f, Loader=SafeLoader)
        
        self.labels = data_yaml['names']
        self.nc = data_yaml['nc']
        
        # Generate random colors for each class
        self.colors = self.generate_colors(self.nc)
        
        # Load YOLO model
        self.yolo = cv2.dnn.readNetFromONNX(onnx_model)
        self.yolo.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.yolo.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
    
    def prediction(self, image):
        row, col, d = image.shape
        
        # Convert the image to square shape (to match YOLO input requirements)
        max_rc = max(row, col)
        input_image = np.zeros((max_rc, max_rc, 3), dtype=np.uint8)
        input_image[0:row, 0:col] = image
        
        # Prepare input blob for YOLO model
        INPUT_WH_YOLO = 640
        blob = cv2.dnn.blobFromImage(input_image, 1/255, (INPUT_WH_YOLO, INPUT_WH_YOLO), swapRB=True, crop=False)
        self.yolo.setInput(blob)
        preds = self.yolo.forward()
        
        # Extract predictions and apply Non-Maximum Suppression (NMS)
        detections = preds[0]
        boxes, confidences, classes = [], [], []
        
        image_h, image_w = input_image.shape[:2]
        x_factor = image_w / INPUT_WH_YOLO
        y_factor = image_h / INPUT_WH_YOLO
        
        for i in range(len(detections)):
            row = detections[i]
            confidence = row[4]
            if confidence > 0.4:  # Confidence threshold
                class_score = row[5:].max()
                class_id = row[5:].argmax()
                
                if class_score > 0.25:  # Class score threshold
                    cx, cy, w, h = row[0:4]
                    left = int((cx - 0.5 * w) * x_factor)
                    top = int((cy - 0.5 * h) * y_factor)
                    width = int(w * x_factor)
                    height = int(h * y_factor)
                    
                    # Clamp bounding box to fit within the image dimensions
                    left = max(0, left)
                    top = max(0, top)
                    width = min(width, image_w - left)
                    height = min(height, image_h - top)
                    
                    box = np.array([left, top, width, height])
                    
                    # Append detected objects
                    confidences.append(float(confidence))
                    boxes.append(box)
                    classes.append(class_id)
        
        # Perform Non-Maximum Suppression (NMS)
        boxes_np = np.array(boxes).tolist()
        confidences_np = np.array(confidences).tolist()
        
        index = cv2.dnn.NMSBoxes(boxes_np, confidences_np, 0.25, 0.45)
        if len(index) > 0:
            index = index.flatten()
            
            # Draw bounding boxes and labels for detected objects
            for ind in index:
                x, y, w, h = boxes_np[ind]
                bb_conf = int(confidences_np[ind] * 100)
                class_id = classes[ind]
                class_name = self.labels[class_id]
                colors = self.colors[class_id]
                text = f'{class_name}: {bb_conf}%'
                
                # Adjust the text size and label background
                text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)[0]  # Reduce text size
                text_w, text_h = text_size[0], text_size[1]
                label_y = max(0, y - text_h - 10)  # Ensure the label doesn't go off the top edge
                
                # Draw bounding box
                cv2.rectangle(image, (x, y), (x + w, y + h), colors, 1)  # Thinner bounding box
                # Draw label background
                cv2.rectangle(image, (x, label_y), (x + min(w, text_w), label_y + text_h + 5), colors, -1)
                # Draw label text
                cv2.putText(image, text, (x, label_y + text_h), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)  # Smaller text
    
        return image
    
    def generate_colors(self, nc):
        # Generate unique colors for each class
        np.random.seed(42)
        return np.random.randint(100, 255, size=(nc, 3)).tolist()

# Code to perform live YOLO prediction on webcam feed
def live_prediction():
    # Initialize YOLO model with ONNX model path and data.yaml file
    yolo = YOLO_Pred('./Model/weights/best.onnx', 'data.yaml')

    # Start video capture from the webcam
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        exit()

    # Set full-screen mode for display
    cv2.namedWindow('Live Prediction', cv2.WND_PROP_FULLSCREEN)
    cv2.setWindowProperty('Live Prediction', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

    # Loop to capture frames from the webcam
    while True:
        # Capture frame-by-frame from the webcam
        ret, frame = cap.read()

        if not ret:
            print("Error: Failed to capture frame.")
            break

        # Perform YOLO prediction on the current frame
        img_pred = yolo.prediction(frame)

        # Display the prediction results
        cv2.imshow('Live Prediction', img_pred)

        # Break the loop when 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the webcam and close all windows
    cap.release()
    cv2.destroyAllWindows()

# Run the live prediction function
if __name__ == "__main__":
    live_prediction()


In [9]:
import cv2
import numpy as np
import yaml
from yaml.loader import SafeLoader

class YOLO_Pred():
    def __init__(self, onnx_model, data_yaml):
        with open(data_yaml, mode='r') as f:
            data_yaml = yaml.load(f, Loader=SafeLoader)

        self.labels = data_yaml['names']
        self.nc = data_yaml['nc']
        self.colors = self.generate_colors(self.nc)
        self.yolo = cv2.dnn.readNetFromONNX(onnx_model)
        self.yolo.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.yolo.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

    def prediction(self, image):
        row, col, d = image.shape
        max_rc = max(row, col)
        input_image = np.zeros((max_rc, max_rc, 3), dtype=np.uint8)
        input_image[0:row, 0:col] = image
        
        INPUT_WH_YOLO = 640
        blob = cv2.dnn.blobFromImage(input_image, 1/255, (INPUT_WH_YOLO, INPUT_WH_YOLO), swapRB=True, crop=False)
        self.yolo.setInput(blob)
        preds = self.yolo.forward()

        detections = preds[0]
        boxes, confidences, classes = [], [], []
        
        image_h, image_w = input_image.shape[:2]
        x_factor = image_w / INPUT_WH_YOLO
        y_factor = image_h / INPUT_WH_YOLO

        for i in range(len(detections)):
            row = detections[i]
            confidence = row[4]
            if confidence > 0.3:  # Lowered confidence threshold to capture more detections
                class_score = row[5:].max()
                class_id = row[5:].argmax()

                if class_score > 0.25:  # Class score threshold
                    cx, cy, w, h = row[0:4]

                    # Scale down the bounding box
                    scale_factor = 0.5  # Adjust this value for size reduction
                    w *= scale_factor
                    h *= scale_factor
                    
                    left = int((cx - 0.5 * w) * x_factor)
                    top = int((cy - 0.5 * h) * y_factor)
                    width = int(w * x_factor)
                    height = int(h * y_factor)

                    # Adjust bounding box size with padding
                    padding = 5
                    left = max(0, left + padding)
                    top = max(0, top + padding)
                    width = min(width - 2 * padding, image_w - left)
                    height = min(height - 2 * padding, image_h - top)

                    box = np.array([left, top, width, height])
                    confidences.append(float(confidence))
                    boxes.append(box)
                    classes.append(class_id)

        # Apply Non-Maximum Suppression (NMS)
        boxes_np = np.array(boxes).tolist()
        confidences_np = np.array(confidences).tolist()

        nms_threshold = 0.4  # Adjust NMS threshold to allow more detections
        index = cv2.dnn.NMSBoxes(boxes_np, confidences_np, 0.3, nms_threshold)

        if len(index) > 0:
            index = index.flatten()
            for ind in index:
                x, y, w, h = boxes_np[ind]
                class_id = classes[ind]
                bb_conf = int(confidences_np[ind] * 100)
                class_name = self.labels[class_id]
                colors = self.colors[class_id]
                text = f'{class_name}: {bb_conf}%'

                # Adjust text size and label background
                text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_PLAIN, 0.5, 1)[0]
                text_w, text_h = text_size[0], text_size[1]
                label_y = max(0, y - text_h - 10)

                # Draw bounding box
                cv2.rectangle(image, (x, y), (x + w, y + h), colors, 2)
                # Draw label background
                cv2.rectangle(image, (x, label_y), (x + min(w, text_w), label_y + text_h + 5), colors, -1)
                # Draw label text
                cv2.putText(image, text, (x, label_y + text_h), cv2.FONT_HERSHEY_PLAIN, 0.5, (0, 0, 0), 1)

        return image

    def generate_colors(self, nc):
        np.random.seed(42)
        return np.random.randint(100, 255, size=(nc, 3)).tolist()

def live_prediction():
    yolo = YOLO_Pred('./Model/weights/best.onnx', 'data.yaml')
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        exit()

    # Set full-screen mode for display
    cv2.namedWindow('Live Prediction', cv2.WND_PROP_FULLSCREEN)
    cv2.setWindowProperty('Live Prediction', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Error: Failed to capture frame.")
            break

        img_pred = yolo.prediction(frame)
        cv2.imshow('Live Prediction', img_pred)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    live_prediction()


In [2]:
import cv2
import numpy as np
import yaml
from yaml.loader import SafeLoader

class YOLO_Pred():
    def __init__(self, onnx_model, data_yaml):
        with open(data_yaml, mode='r') as f:
            data_yaml = yaml.load(f, Loader=SafeLoader)

        self.labels = data_yaml['names']
        self.nc = data_yaml['nc']
        self.colors = self.generate_colors(self.nc)
        self.yolo = cv2.dnn.readNetFromONNX(onnx_model)
        self.yolo.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.yolo.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

    def prediction(self, image):
        row, col, d = image.shape
        max_rc = max(row, col)
        input_image = np.zeros((max_rc, max_rc, 3), dtype=np.uint8)
        input_image[0:row, 0:col] = image
        
        INPUT_WH_YOLO = 640
        blob = cv2.dnn.blobFromImage(input_image, 1/255, (INPUT_WH_YOLO, INPUT_WH_YOLO), swapRB=True, crop=False)
        self.yolo.setInput(blob)
        preds = self.yolo.forward()

        detections = preds[0]
        boxes, confidences, classes = [], [], []
        
        image_h, image_w = input_image.shape[:2]
        x_factor = image_w / INPUT_WH_YOLO
        y_factor = image_h / INPUT_WH_YOLO

        # Dictionary to count detected products
        product_count = {}

        for i in range(len(detections)):
            row = detections[i]
            confidence = row[4]
            if confidence > 0.3:  # Lowered confidence threshold to capture more detections
                class_score = row[5:].max()
                class_id = row[5:].argmax()

                if class_score > 0.25:  # Class score threshold
                    cx, cy, w, h = row[0:4]

                    # Scale down the bounding box
                    scale_factor = 0.5  # Adjust this value for size reduction
                    w *= scale_factor
                    h *= scale_factor
                    
                    left = int((cx - 0.5 * w) * x_factor)
                    top = int((cy - 0.5 * h) * y_factor)
                    width = int(w * x_factor)
                    height = int(h * y_factor)

                    # Adjust bounding box size with padding
                    padding = 5
                    left = max(0, left + padding)
                    top = max(0, top + padding)
                    width = min(width - 2 * padding, image_w - left)
                    height = min(height - 2 * padding, image_h - top)

                    box = np.array([left, top, width, height])
                    confidences.append(float(confidence))
                    boxes.append(box)
                    classes.append(class_id)

                    # Update product count
                    class_name = self.labels[class_id]
                    product_count[class_name] = product_count.get(class_name, 0) + 1

        # Apply Non-Maximum Suppression (NMS)
        boxes_np = np.array(boxes).tolist()
        confidences_np = np.array(confidences).tolist()

        nms_threshold = 0.4  # Adjust NMS threshold to allow more detections
        index = cv2.dnn.NMSBoxes(boxes_np, confidences_np, 0.3, nms_threshold)

        if len(index) > 0:
            index = index.flatten()
            for ind in index:
                x, y, w, h = boxes_np[ind]
                class_id = classes[ind]
                bb_conf = int(confidences_np[ind] * 100)
                class_name = self.labels[class_id]
                colors = self.colors[class_id]
                text = f'{class_name}: {bb_conf}%'

                # Adjust text size and label background
                text_size = cv2.getTextSize(text, cv2.FONT_HERSHEY_PLAIN, 0.5, 1)[0]
                text_w, text_h = text_size[0], text_size[1]
                label_y = max(0, y - text_h - 10)

                # Draw bounding box
                cv2.rectangle(image, (x, y), (x + w, y + h), colors, 2)
                # Draw label background
                cv2.rectangle(image, (x, label_y), (x + min(w, text_w), label_y + text_h + 5), colors, -1)
                # Draw label text
                cv2.putText(image, text, (x, label_y + text_h), cv2.FONT_HERSHEY_PLAIN, 0.5, (0, 0, 0), 1)

        # Display product count in the top left corner of the screen
        for i, (product, count) in enumerate(product_count.items()):
            text = f'{product}: {count}'
            cv2.putText(image, text, (10, 30 + i * 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        return image

    def generate_colors(self, nc):
        np.random.seed(42)
        return np.random.randint(100, 255, size=(nc, 3)).tolist()

def live_prediction():
    yolo = YOLO_Pred('./Model/weights/best.onnx', 'data.yaml')
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        exit()

    # Set full-screen mode for display
    cv2.namedWindow('Live Prediction', cv2.WND_PROP_FULLSCREEN)
    cv2.setWindowProperty('Live Prediction', cv2.WND_PROP_FULLSCREEN, cv2.WINDOW_FULLSCREEN)

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Error: Failed to capture frame.")
            break

        img_pred = yolo.prediction(frame)
        cv2.imshow('Live Prediction', img_pred)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    live_prediction()


In [5]:
import cv2
import numpy as np
import yaml
from yaml.loader import SafeLoader

class YOLO_Pred():
    def __init__(self, onnx_model, data_yaml):
        with open(data_yaml, mode='r') as f:
            data_yaml = yaml.load(f, Loader=SafeLoader)

        self.labels = data_yaml['names']
        self.nc = data_yaml['nc']
        self.colors = self.generate_colors(self.nc)

        # Load the YOLO ONNX model
        self.yolo = cv2.dnn.readNetFromONNX(onnx_model)
        self.yolo.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.yolo.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

    def prediction(self, image):
        # Resize the image to a square and create input blob
        input_image = self.preprocess_image(image)
        INPUT_WH_YOLO = 640

        # Convert image to a blob as expected by YOLO model
        blob = cv2.dnn.blobFromImage(input_image, 1/255, (INPUT_WH_YOLO, INPUT_WH_YOLO), swapRB=True, crop=False)
        self.yolo.setInput(blob)

        # Forward pass through the network
        preds = self.yolo.forward()

        # Check if the model returned any predictions
        if preds is None or len(preds) == 0:
            print("No predictions were made by the model.")
            return image

        # Debugging output: print prediction shapes
        print(f"Predictions shape: {preds[0].shape}")

        # Process the predictions and draw boxes
        return self.process_predictions(preds[0], image)

    def preprocess_image(self, image):
        # Adjust the image to a square size
        row, col, _ = image.shape
        max_rc = max(row, col)
        input_image = np.zeros((max_rc, max_rc, 3), dtype=np.uint8)
        input_image[0:row, 0:col] = image
        return input_image

    def process_predictions(self, detections, image):
        INPUT_WH_YOLO = 640
        image_h, image_w = image.shape[:2]
        x_factor = image_w / INPUT_WH_YOLO
        y_factor = image_h / INPUT_WH_YOLO

        boxes, confidences, classes = [], [], []

        for detection in detections:
            confidence = detection[4]
            if confidence > 0.5:  # Confidence threshold
                class_scores = detection[5:]
                class_id = np.argmax(class_scores)
                class_score = class_scores[class_id]

                if class_score > 0.4:  # Class score threshold
                    cx, cy, w, h = detection[:4]

                    # Rescale bounding box coordinates
                    left = int((cx - 0.5 * w) * x_factor)
                    top = int((cy - 0.5 * h) * y_factor)
                    width = int(w * x_factor)
                    height = int(h * y_factor)

                    boxes.append([left, top, width, height])
                    confidences.append(float(confidence))
                    classes.append(class_id)

        # Apply Non-Maximum Suppression (NMS) to remove redundant boxes
        indices = cv2.dnn.NMSBoxes(boxes, confidences, score_threshold=0.5, nms_threshold=0.4)

        if len(indices) > 0:
            for i in indices.flatten():
                x, y, w, h = boxes[i]
                class_id = classes[i]
                color = self.colors[class_id]
                label = f"{self.labels[class_id]}: {int(confidences[i] * 100)}%"

                # Draw bounding box with color
                cv2.rectangle(image, (x, y), (x + w, y + h), color, 2)
                # Draw label
                cv2.putText(image, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        return image

    def generate_colors(self, nc):
        np.random.seed(42)
        return np.random.randint(100, 255, size=(nc, 3)).tolist()


def live_prediction():
    # Replace with your ONNX model and YAML file paths
    yolo = YOLO_Pred('./Model/weights/best.onnx', './data.yaml')
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error: Failed to capture frame.")
            break

        # Run the YOLO prediction
        img_pred = yolo.prediction(frame)

        # Display the prediction
        cv2.imshow('YOLO Prediction', img_pred)

        # Exit loop on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    live_prediction()


Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)
Predictions shape: (25200, 13)


In [ ]:
import cv2
import numpy as np
import yaml
from yaml.loader import SafeLoader

class YOLO_Pred():
    def __init__(self, onnx_model, data_yaml):
        with open(data_yaml, mode='r') as f:
            data_yaml = yaml.load(f, Loader=SafeLoader)

        self.labels = data_yaml['names']
        self.nc = data_yaml['nc']
        self.colors = self.generate_colors(self.nc)

        # Load the YOLO ONNX model
        self.yolo = cv2.dnn.readNetFromONNX(onnx_model)
        self.yolo.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.yolo.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

    def prediction(self, image):
        # Resize the image to a square and create input blob
        input_image = self.preprocess_image(image)
        INPUT_WH_YOLO = 640

        # Convert image to a blob as expected by YOLO model
        blob = cv2.dnn.blobFromImage(input_image, 1/255, (INPUT_WH_YOLO, INPUT_WH_YOLO), swapRB=True, crop=False)
        self.yolo.setInput(blob)

        # Forward pass through the network
        preds = self.yolo.forward()

        # Check if the model returned any predictions
        if preds is None or len(preds) == 0:
            print("No predictions were made by the model.")
            return image

        # Debugging output: print prediction shapes
        print(f"Predictions shape: {preds[0].shape}")

        # Process the predictions and draw boxes
        return self.process_predictions(preds[0], image)

    def preprocess_image(self, image):
        # Adjust the image to a square size
        row, col, _ = image.shape
        max_rc = max(row, col)
        input_image = np.zeros((max_rc, max_rc, 3), dtype=np.uint8)
        input_image[0:row, 0:col] = image
        return input_image

    def process_predictions(self, detections, image):
        INPUT_WH_YOLO = 640
        image_h, image_w = image.shape[:2]
        x_factor = image_w / INPUT_WH_YOLO
        y_factor = image_h / INPUT_WH_YOLO

        boxes, confidences, classes = [], [], []

        for detection in detections:
            confidence = detection[4]
            if confidence > 0.5:  # Confidence threshold
                class_scores = detection[5:]
                class_id = np.argmax(class_scores)
                class_score = class_scores[class_id]

                if class_score > 0.4:  # Class score threshold
                    cx, cy, w, h = detection[:4]

                    # Rescale bounding box coordinates
                    left = int((cx - 0.5 * w) * x_factor)
                    top = int((cy - 0.5 * h) * y_factor)
                    width = int(w * x_factor)
                    height = int(h * y_factor)

                    boxes.append([left, top, width, height])
                    confidences.append(float(confidence))
                    classes.append(class_id)

        # Apply Non-Maximum Suppression (NMS) to remove redundant boxes
        indices = cv2.dnn.NMSBoxes(boxes, confidences, score_threshold=0.5, nms_threshold=0.4)

        if len(indices) > 0:
            for i in indices.flatten():
                x, y, w, h = boxes[i]
                class_id = classes[i]
                color = self.colors[class_id]
                label = f"{self.labels[class_id]}: {int(confidences[i] * 100)}%"

                # Draw bounding box with color
                cv2.rectangle(image, (x, y), (x + w, y + h), color, 2)
                # Draw label
                cv2.putText(image, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        return image

    def generate_colors(self, nc):
        np.random.seed(42)
        return np.random.randint(100, 255, size=(nc, 3)).tolist()


def predict_from_image(image_path):
    # Replace with your ONNX model and YAML file paths
    yolo = YOLO_Pred('./Model/weights/best.onnx', './data.yaml')
    
    # Load the image from the provided path
    image = cv2.imread(image_path)

    if image is None:
        print(f"Error: Could not load image at {image_path}")
        return

    # Run the YOLO prediction
    img_pred = yolo.prediction(image)

    # Display the prediction
    cv2.imshow('YOLO Prediction', img_pred)

    # Save the output image with predictions
    output_path = "output_prediction.jpg"
    cv2.imwrite(output_path, img_pred)
    print(f"Prediction saved to {output_path}")

    # Wait for a key press to close the image window
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    # Replace with the path to the image file you want to process
    image_path = "sample.jpg"
    predict_from_image(image_path)


In [ ]:
import cv2
import numpy as np
import yaml
from yaml.loader import SafeLoader

class YOLO_Pred():
    def __init__(self, onnx_model, data_yaml):
        with open(data_yaml, mode='r') as f:
            data_yaml = yaml.load(f, Loader=SafeLoader)

        self.labels = data_yaml['names']
        self.nc = data_yaml['nc']
        self.colors = self.generate_colors(self.nc)

        # Load the YOLO ONNX model
        self.yolo = cv2.dnn.readNetFromONNX(onnx_model)
        self.yolo.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.yolo.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

    def prediction(self, image):
        # Resize the image to a square and create input blob
        input_image = self.preprocess_image(image)
        INPUT_WH_YOLO = 640

        # Convert image to a blob as expected by YOLO model
        blob = cv2.dnn.blobFromImage(input_image, 1/255, (INPUT_WH_YOLO, INPUT_WH_YOLO), swapRB=True, crop=False)
        self.yolo.setInput(blob)

        # Forward pass through the network
        preds = self.yolo.forward()

        # Check if the model returned any predictions
        if preds is None or len(preds) == 0:
            print("No predictions were made by the model.")
            return image, 0

        # Debugging output: print prediction shapes
        print(f"Predictions shape: {preds[0].shape}")

        # Process the predictions and draw boxes
        return self.process_predictions(preds[0], image)

    def preprocess_image(self, image):
        # Adjust the image to a square size
        row, col, _ = image.shape
        max_rc = max(row, col)
        input_image = np.zeros((max_rc, max_rc, 3), dtype=np.uint8)
        input_image[0:row, 0:col] = image
        return input_image

    def process_predictions(self, detections, image):
        INPUT_WH_YOLO = 640
        image_h, image_w = image.shape[:2]
        x_factor = image_w / INPUT_WH_YOLO
        y_factor = image_h / INPUT_WH_YOLO

        boxes, confidences, classes = [], [], []

        for detection in detections:
            confidence = detection[4]
            if confidence > 0.6:  # Increased confidence threshold
                class_scores = detection[5:]
                class_id = np.argmax(class_scores)
                class_score = class_scores[class_id]

                if class_score > 0.5:  # Increased class score threshold
                    cx, cy, w, h = detection[:4]

                    # Rescale bounding box coordinates
                    left = int((cx - 0.5 * w) * x_factor)
                    top = int((cy - 0.5 * h) * y_factor)
                    width = int(w * x_factor)
                    height = int(h * y_factor)

                    boxes.append([left, top, width, height])
                    confidences.append(float(confidence))
                    classes.append(class_id)

        # Apply Non-Maximum Suppression (NMS) to remove redundant boxes
        indices = cv2.dnn.NMSBoxes(boxes, confidences, score_threshold=0.5, nms_threshold=0.3)

        detected_packs = len(indices)  # Count only after applying NMS

        if len(indices) > 0:
            for i in indices.flatten():
                x, y, w, h = boxes[i]
                class_id = classes[i]
                color = self.colors[class_id]
                label = f"{self.labels[class_id]}: {int(confidences[i] * 100)}%"

                # Draw bounding box with color
                cv2.rectangle(image, (x, y), (x + w, y + h), color, 2)
                # Draw label
                cv2.putText(image, label, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        return image, detected_packs

    def generate_colors(self, nc):
        np.random.seed(42)
        return np.random.randint(100, 255, size=(nc, 3)).tolist()


def predict_and_count_packs(image_path):
    # Replace with your ONNX model and YAML file paths
    yolo = YOLO_Pred('./Model/weights/best.onnx', './data.yaml')
    
    # Load the image from the provided path
    image = cv2.imread(image_path)

    if image is None:
        print(f"Error: Could not load image at {image_path}")
        return

    # Run the YOLO prediction and count the detected packs
    img_pred, pack_count = yolo.prediction(image)

    # Display the prediction and pack count
    cv2.putText(img_pred, f"Detected Packs: {pack_count}", (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('YOLO Prediction', img_pred)

    # Save the output image with predictions and pack count
    output_path = "output_with_count.jpg"
    cv2.imwrite(output_path, img_pred)
    print(f"Prediction saved to {output_path}, Packs Count: {pack_count}")

    # Wait for a key press to close the image window
    cv2.waitKey(0)
    cv2.destroyAllWindows()


if __name__ == "__main__":
    # Replace with the path to the image file you want to process
    image_path = "pack.jpg"
    predict_and_count_packs(image_path)


In [3]:
import cv2
import numpy as np
import yaml
from yaml.loader import SafeLoader

class YOLO_Pred():
    def __init__(self, onnx_model, data_yaml):
        with open(data_yaml, mode='r') as f:
            data_yaml = yaml.load(f, Loader=SafeLoader)

        self.labels = data_yaml['names']
        self.nc = data_yaml['nc']
        self.colors = self.generate_colors(self.nc)
        self.yolo = cv2.dnn.readNetFromONNX(onnx_model)
        self.yolo.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        self.yolo.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)
        self.object_tracker = {}  # To keep track of detected objects

    def prediction(self, image):
        row, col, d = image.shape
        max_rc = max(row, col)
        input_image = np.zeros((max_rc, max_rc, 3), dtype=np.uint8)
        input_image[0:row, 0:col] = image

        INPUT_WH_YOLO = 640
        blob = cv2.dnn.blobFromImage(input_image, 1/255, (INPUT_WH_YOLO, INPUT_WH_YOLO), swapRB=True, crop=False)
        self.yolo.setInput(blob)
        preds = self.yolo.forward()

        detections = preds[0]
        boxes, confidences, classes = [], [], []

        image_h, image_w = input_image.shape[:2]
        x_factor = image_w / INPUT_WH_YOLO
        y_factor = image_h / INPUT_WH_YOLO

        for i in range(len(detections)):
            row = detections[i]
            confidence = row[4]
            if confidence > 0.3:
                class_score = row[5:].max()
                class_id = row[5:].argmax()

                if class_score > 0.25:
                    cx, cy, w, h = row[0:4]
                    w *= 0.5  # Adjust bounding box scale factor
                    h *= 0.5
                    
                    left = int((cx - 0.5 * w) * x_factor)
                    top = int((cy - 0.5 * h) * y_factor)
                    width = int(w * x_factor)
                    height = int(h * y_factor)

                    padding = 5
                    left = max(0, left + padding)
                    top = max(0, top + padding)
                    width = min(width - 2 * padding, image_w - left)
                    height = min(height - 2 * padding, image_h - top)

                    box = np.array([left, top, width, height])
                    confidences.append(float(confidence))
                    boxes.append(box)
                    classes.append(class_id)

        boxes_np = np.array(boxes).tolist()
        confidences_np = np.array(confidences).tolist()

        # Apply NMS to remove duplicate detections
        nms_threshold = 0.4
        index = cv2.dnn.NMSBoxes(boxes_np, confidences_np, 0.3, nms_threshold)

        object_count = 0  # Variable to count unique objects

        if len(index) > 0:
            index = index.flatten()
            for ind in index:
                x, y, w, h = boxes_np[ind]
                class_id = classes[ind]
                bb_conf = int(confidences_np[ind] * 100)
                class_name = self.labels[class_id]
                colors = self.colors[class_id]
                text = f'{class_name}: {bb_conf}%'

                # Track the objects with their bounding box
                object_id = f"{x}_{y}_{w}_{h}"
                if object_id not in self.object_tracker:
                    self.object_tracker[object_id] = True  # Add to tracker
                    object_count += 1  # Increment count for new objects

                # Draw bounding box
                cv2.rectangle(image, (x, y), (x + w, y + h), colors, 2)
                cv2.putText(image, text, (x, y - 10), cv2.FONT_HERSHEY_PLAIN, 1, (0, 0, 0), 2)

        # Display the count on the screen
        cv2.putText(image, f"Product Count: {object_count}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)
        
        return image

    def generate_colors(self, nc):
        np.random.seed(42)
        return np.random.randint(100, 255, size=(nc, 3)).tolist()

def live_prediction():
    yolo = YOLO_Pred('./Model/weights/best.onnx', 'data.yaml')
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        exit()

    while True:
        ret, frame = cap.read()

        if not ret:
            print("Error: Failed to capture frame.")
            break

        img_pred = yolo.prediction(frame)
        cv2.imshow('Live Prediction', img_pred)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    live_prediction()


FileNotFoundError: [Errno 2] No such file or directory: 'data.yaml'